Import required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

Load project utilities & Initialize notebook

In [0]:
%run /Workspace/project1/setup_catalogs/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', 'project1_cat', 'catalog')
dbutils.widgets.text('data_source', 'gross_price', 'data_source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
print(catalog, data_source)

In [0]:
base_path = f's3://sportsbar-childpro/{data_source}/*.csv'
print(base_path)

### Bronze

In [0]:
df = spark.read.format('csv')\
        .option('header', 'true')\
        .option('inferSchema', 'true')\
        .load(base_path)\
        .withColumn('read_timestamp', F.current_timestamp())

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

### Silver

In [0]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')
display(df_bronze)

1. Normalise the month field

In [0]:
display(df_bronze.select('month').distinct())

In [0]:
# date formatting
date_formats = ['yyyy/MM/dd', 'dd/MM/yyyy', 'yyyy-MM-dd', 'dd-MM-yyyy']

df_silver = df_bronze.withColumn('month', 
                    F.coalesce(
                    F.try_to_date(F.col('month'), 'yyyy/MM/dd'),
                    F.try_to_date(F.col('month'), 'dd/MM/yyyy'),
                    F.try_to_date(F.col('month'), 'yyyy-MM-dd'),
                    F.try_to_date(F.col('month'), 'dd-MM-yyyy')
                    ))

In [0]:
display(df_silver.select('month').distinct())

2. Handling gross price

In [0]:
display(df_silver.limit(10))

In [0]:
# We are validating the gross_price column, converting only valid numeric values to double, fixing negative price by making them positive, and replacing all non-numeric values with 0

display(df_silver.select('gross_price'))

In [0]:
df_silver = df_silver.withColumn('gross_price', 
                                 F.when(F.col('gross_price').rlike(r'^-?\d+(\.\d+)?$'), 
                                        F.when(F.col('gross_price').cast('double') < 0, -1 * F.col('gross_price').cast('double')).otherwise(F.col('gross_price').cast('double')))
                                 .otherwise(0))

In [0]:
df_silver.show()

In [0]:
# We are enrich the silver dataset by processing an inner join with the products table to fetch the correct product_code for each product_id.

df_products = spark.table('project1_cat.silver.products')
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on='product_id', how='inner')
df_joined = df_joined.select('product_id', 'product_code', 'month', 'gross_price', 'read_timestamp')
display(df_joined.limit(10))

In [0]:
df_joined.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.option('mergeSchema', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

### Gold

In [0]:
df_silver = spark.sql(f'select * from {catalog}.{silver_schema}.{data_source}')

In [0]:
df_gold = df_silver.select('product_code', 'month', 'gross_price')
display(df_gold.limit(10))

In [0]:
df_gold.write\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.option('mergeSchema', 'true')\
.mode('overwrite')\
.saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

### Merging data source with parent

In [0]:
df_gold_price = spark.table('project1_cat.gold.sb_dim_gross_price')
display(df_gold_price)

Get the price for each product_code (aggregated by year)

In [0]:
df_gold_price = (df_gold_price.withColumn('year', F.year('month'))\
                 .withColumn('is_zero', F.when(F.col('gross_price') == 0, 1).otherwise(0)))

w = Window.partitionBy('product_code', 'year').orderBy(F.col('is_zero'),F.col('month').desc())

df_gold_latest_price = (df_gold_price.withColumn('rnk', F.row_number().over(w)).filter(F.col('rnk') == 1))


In [0]:
display(df_gold_latest_price.limit(10))

In [0]:
# Required columns

df_gold_latest_price = df_gold_latest_price.select('product_code', 'year', 'gross_price').withColumnRenamed('gross_price', 'price_inr').select('product_code', 'price_inr', 'year')

df_gold_latest_price = df_gold_latest_price.withColumn('year', F.col('year').cast('string'))

display(df_gold_latest_price)

In [0]:
df_gold_latest_price.printSchema()

In [0]:
delta_table = DeltaTable.forName(spark, 'project1_cat.gold.dim_gross_price')


delta_table.alias('target').merge(
    source=df_gold_latest_price.alias('source'),
    condition=F.expr('target.product_code = source.product_code AND target.year = source.year'))\
.whenMatchedUpdate(
    set={'price_inr': F.col('source.price_inr'),
         'year': F.col('source.year')}
).whenNotMatchedInsert(
    values = {'product_code': F.col('source.product_code'),
             'price_inr': F.col('source.price_inr'),
             'year': F.col('source.year')}
).execute()